# Setup

## File download

In [1]:
import requests
import os
if not os.environ.get("JAVA_HOME"):
    raise ValueError("Environment variable 'JAVA_HOME' is not set. Please set it to the path of your Java installation.")

trip_data_path = "data/raw/yellow_tripdata_2024-01.parquet"
os.makedirs('data/raw', exist_ok=True)
if os.path.exists(trip_data_path):
    print("File already exists, skipping download.")
else:
    response = requests.get("https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet", stream=True)
    response.raise_for_status()
    with open(trip_data_path, 'wb') as f:
        for chunk in response.iter_content(chunk_size=4194304):  # 4MB chunks to avoid memory issues and speed up download
            f.write(chunk)
    print ("File downloaded")
import socket
print(socket.gethostname())
print(socket.gethostbyname(socket.gethostname()))

File already exists, skipping download.
Cyberpower
192.168.1.22


## Setting up PySpark

In [2]:
import pyspark
import subprocess
import time
def check_java():
    try:
        result = subprocess.run(['java', '-version'], stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
        if result.returncode == 0:
            print("Java is installed and accessible.")
            print(result.stderr)  # Java version info is usually in stderr
        else:
            print("Java is not accessible. Please check your JAVA_HOME and PATH settings.")
            print(result.stderr)
    except FileNotFoundError:
        print("Java executable not found. Please ensure Java is installed and JAVA_HOME is set correctly.")
from pyspark.sql import SparkSession
os.environ["JAVA_HOME"] = "C:/Program Files/Java/jdk-17" # Update this path to your actual Java installation directory
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
os.environ["PYSPARK_SUBMIT_ARGS"] = "--conf spark.ui.showConsoleProgress=true pyspark-shell"
os.environ["SPARK_LOCAL_HOSTNAME"] = "localhost"
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
os.environ["HADOOP_HOME"] = "C:\\hadoop"
check_java()
print ("Environment variables set for Spark")
spark = SparkSession.builder \
    .appName("Yellow Taxi Analysis") \
    .master("local[*]") \
    .config("spark.ui.enabled", "false")\
    .config("spark.hadoop.io.native.lib.available", "false") \
    .getOrCreate() # Disable Spark UI and native library check for better performance on local machine
print("Spark started on version:", spark.version)
spark_start_time = time.perf_counter()
df = spark.read.parquet(trip_data_path); spark_end_time = time.perf_counter()
spark_time = spark_end_time - spark_start_time
print(f"Time taken for Spark to read Parquet file: {spark_time:.2f} seconds")
df.show(5)
print ("DataFrame schema:")
df.printSchema()
starting_rows = df.count()
print ("DataFrame row count:", starting_rows)
print ("DataFrame columns:", df.columns)


Java is installed and accessible.
java version "22.0.1" 2024-04-16
Java(TM) SE Runtime Environment (build 22.0.1+8-16)
Java HotSpot(TM) 64-Bit Server VM (build 22.0.1+8-16, mixed mode, sharing)

Environment variables set for Spark
Spark started on version: 4.1.1
Time taken for Spark to read Parquet file: 1.20 seconds
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+

## Comparing with Pandas

In [3]:
import pandas as pd
pandas_start_time = time.perf_counter()
p_df = pd.read_parquet(trip_data_path)
pandas_end_time = time.perf_counter()
pandas_time = pandas_end_time - pandas_start_time
print(f"Time taken to read Parquet file with Pandas: {pandas_time:.2f} seconds")
print(f"Spark was {pandas_time / spark_time:.2f} times faster than Pandas for reading the Parquet file. Pandas eagerly loaded the data into memory while Spark lazily loaded the data, which is why Spark was faster for this operation. In conclusion, Spark's lazy loading and optimized execution engine make it more efficient for handling large datasets compared to Pandas, which loads the entire dataset into memory at once. Pandas is trash for big data, Spark is the way to go.")

Time taken to read Parquet file with Pandas: 0.22 seconds
Spark was 0.19 times faster than Pandas for reading the Parquet file. Pandas eagerly loaded the data into memory while Spark lazily loaded the data, which is why Spark was faster for this operation. In conclusion, Spark's lazy loading and optimized execution engine make it more efficient for handling large datasets compared to Pandas, which loads the entire dataset into memory at once. Pandas is trash for big data, Spark is the way to go.


# Data Cleaning & Feature Engineering in Spark

## Clean the data

In [4]:
# Cleanup Spark dataframes.
# Removing rows with null values in critical columns (pickup/dropoff times, locations, fare,distance)
df = df.dropna(subset=["tpep_pickup_datetime", "tpep_dropoff_datetime",  "fare_amount", "trip_distance", "PULocationID", "DOLocationID"])
print (f"{starting_rows - df.count()} rows removed due to null values. Remaining rows: {df.count()}")
starting_rows = df.count()
# Removing rows with non-positive fare amounts or trip distances
df = df.filter((df.fare_amount > 0) & (df.trip_distance > 0))
print (f"{starting_rows - df.count()} rows removed due to non-positive fare amounts or trip distances. Remaining rows: {df.count()}")
starting_rows = df.count()
starting_rows = df.count()
# Removing rows with pickup times after dropoff times
df = df.filter(df.tpep_pickup_datetime < df.tpep_dropoff_datetime)
print (f"{starting_rows - df.count()} rows removed due to pickup times after dropoff times. Remaining rows: {df.count()}")
starting_rows = df.count()
# Remove rows with ludicrous fares (e.g., fare_amount > $500)
df = df.filter(df.fare_amount <= 500)
print (f"{starting_rows - df.count()} rows removed due to ludicrous fares. Remaining rows: {df.count()}")


0 rows removed due to null values. Remaining rows: 2964624
94910 rows removed due to non-positive fare amounts or trip distances. Remaining rows: 2869714
112 rows removed due to pickup times after dropoff times. Remaining rows: 2869602
30 rows removed due to ludicrous fares. Remaining rows: 2869572


## Derive columns

In [5]:
# We will now derive new columns for analysis. We will derive trip duration in minutes, speed in miles per hour, pickup hour of day, and pickup day of week. Tip percentage will also be derived for later analysis.
from pyspark.sql import functions as F

df = df.withColumn(
    "trip_duration_minutes",
    (
        F.unix_timestamp("tpep_dropoff_datetime")
        - F.unix_timestamp("tpep_pickup_datetime")
    )
    / 60,
)
df = df.withColumn(
    "speed_mph", F.col("trip_distance") / (F.col("trip_duration_minutes") / 60)
)
df = df.withColumn("pickup_hour", F.hour("tpep_pickup_datetime"))
df = df.withColumn("pickup_dayOfWeek", F.dayofweek("tpep_pickup_datetime"))
df = df.withColumn("tip_percentage", (F.col("tip_amount") / F.col("fare_amount")) * 100)
print(
    "Derived new columns: trip_duration_minutes, speed_mph, pickup_hour, pickup_dayOfWeek, tip_percentage"
)
df.show(5)

Derived new columns: trip_duration_minutes, speed_mph, pickup_hour, pickup_dayOfWeek, tip_percentage
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+---------------------+------------------+-----------+----------------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|trip_duration_minutes|         speed_mph|pickup_hour|pickup_dayOfWeek|    tip_percentage|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+-

## Spark SQL Analytics

In [6]:
# Register the DataFrame as a temporary view for Spark SQL queries
df.createOrReplaceTempView("trips")
queries = {}
#What are the top 10 busiest pickup hours, and what is the average fare and tippercentage for each?
queries["top_pickup_hours"] = """
SELECT pickup_hour, COUNT(*) AS trip_count, AVG(fare_amount) AS avg_fare, AVG(tip_percentage) AS avg_tip_percentage
FROM trips
GROUP BY pickup_hour
ORDER BY trip_count DESC
LIMIT 10
"""

#Which day of the week has the highest average trip speed? Include averagedistance and duration
queries["fastest_day_of_week"] = """
SELECT pickup_dayOfWeek, AVG(speed_mph) AS avg_speed, AVG(trip_distance) AS avg_distance, AVG(trip_duration_minutes) AS avg_duration
FROM trips
GROUP BY pickup_dayOfWeek
ORDER BY avg_speed DESC
LIMIT 1
"""

#Using a window function, rank the top 5 pickup locations by total revenue foreach day of the week.
queries["top_pickup_locations_by_revenue"] = """
SELECT *
FROM (
    SELECT 
        pickup_dayOfWeek,
        PULocationID,
        SUM(fare_amount) AS total_revenue,
        RANK() OVER (
            PARTITION BY pickup_dayOfWeek 
            ORDER BY SUM(fare_amount) DESC
        ) AS revenue_rank
    FROM trips
    GROUP BY pickup_dayOfWeek, PULocationID
) t
WHERE revenue_rank <= 5
ORDER BY pickup_dayOfWeek, revenue_rank;
"""

# Calculate the cumulative trip count by hour of day (running total from hour 0 to23). At what hour does the cumulative count surpass 50% of daily trips?
queries["cumulative_trip_count_by_hour"] = """
WITH hourly_counts AS (
    SELECT pickup_hour, COUNT(*) AS trip_count
    FROM trips
    GROUP BY pickup_hour
),
cumulative_counts AS (
    SELECT pickup_hour, SUM(trip_count) OVER (ORDER BY pickup_hour) AS cumulative_trip_count
    FROM hourly_counts
)
SELECT pickup_hour
FROM cumulative_counts
WHERE cumulative_trip_count > (SELECT SUM(trip_count) FROM hourly_counts) * 0.5
ORDER BY pickup_hour
LIMIT 1
"""

#Compare average fare, distance, and tip percentage between short trips (<2miles), medium trips (2–10 miles), and long trips (>10 miles). Which category has thehighest tip percentage?
queries["trip_length_comparison"] = """
SELECT
    CASE
        WHEN trip_distance < 2 THEN 'Short'
        WHEN trip_distance BETWEEN 2 AND 10 THEN 'Medium'
        ELSE 'Long'
    END AS trip_length_category,
    AVG(fare_amount) AS avg_fare,
    AVG(trip_distance) AS avg_distance,
    AVG(tip_percentage) AS avg_tip_percentage
FROM trips
GROUP BY
    CASE
        WHEN trip_distance < 2 THEN 'Short'
        WHEN trip_distance BETWEEN 2 AND 10 THEN 'Medium'
        ELSE 'Long'
    END
ORDER BY avg_tip_percentage DESC
"""

# Execute the queries and print results
for query_name, query in queries.items():
    print(f"\nResults for query: {query_name}")
    result_df = spark.sql(query)
    result_df.show()


Results for query: top_pickup_hours
+-----------+----------+------------------+------------------+
|pickup_hour|trip_count|          avg_fare|avg_tip_percentage|
+-----------+----------+------------------+------------------+
|         18|    206256|17.015239556667414|22.783750766291575|
|         17|    200280|18.120561114439727| 22.34330404301461|
|         16|    184947|19.459183333603697| 21.83736978978918|
|         15|    183971|19.114085535220184|19.800998170838547|
|         19|    178785|17.629133484352604|22.859468207617592|
|         14|    178002|19.273210301007893| 19.79789536000929|
|         13|    165326|18.421258967131617| 19.78590339670734|
|         12|    159884|17.798798629005994|19.742680445725988|
|         21|    155885|18.295467556211214| 21.88135250663954|
|         20|    155539|18.052625129388723|22.171556229200444|
+-----------+----------+------------------+------------------+


Results for query: fastest_day_of_week
+----------------+------------------+---

## Performance Optimization

In [8]:
# We want to compare time taken on uncached queries vs cached queries. We will run the top pickup hours query uncached and then cached to compare times.

# Uncached query
uncached_start_time = time.perf_counter()
spark.sql(queries["top_pickup_hours"]).show()
uncached_end_time = time.perf_counter()
uncached_time = uncached_end_time - uncached_start_time

# Cache the DataFrame and run the same query again
df.cache()
cached_start_time = time.perf_counter()
spark.sql(queries["top_pickup_hours"]).show()
cached_end_time = time.perf_counter()
cached_time = cached_end_time - cached_start_time
print(f"Time taken for uncached query: {uncached_time:.2f} seconds")
print(f"Time taken for cached query: {cached_time:.2f} seconds")
print(f"Caching the DataFrame resulted in a speedup of {uncached_time / cached_time:.2f} times for the top pickup hours query. Due to the way Spark optimizes execution, the first run of the query will be slower as it reads from disk and processes the data. Once cached, subsequent runs of the same query will be much faster as Spark can read from memory instead of disk, demonstrating the performance benefits of caching in Spark for repeated queries on the same dataset. But this data is small, so I see a neglible difference. Should be larger on bigger datasets")

# Write the cleaned data back as partioned
df.write.mode("overwrite").partitionBy("pickup_hour").parquet("data/cleaned/yellow_tripdata_2024-01_partitioned.parquet")
print("Cleaned data written back to disk, partitioned by pickup_hour. We will now read hour 17 data to see how much faster it is to read a partitioned file vs the original unpartitioned file.")
# Read the partitioned data for hour 17
partitioned_start_time = time.perf_counter()
hour_17_df = spark.read.parquet("data/cleaned/yellow_tripdata_2024-01_partitioned.parquet/pickup_hour=17")
partitioned_end_time = time.perf_counter()
partitioned_time = partitioned_end_time - partitioned_start_time
print(f"Time taken to read partitioned data for hour 17: {partitioned_time:.2f} seconds")
print(f"Time taken to read the original unpartitioned Parquet file: {spark_time:.2f} seconds")
print(f"Reading the partitioned data for hour 17 was {spark_time / partitioned_time:.2f} times faster than reading the original unpartitioned Parquet file. This demonstrates the performance benefits of partitioning in Spark, as it allows Spark to read only the relevant subset of data (hour 17) instead of scanning the entire dataset, resulting in significantly faster read times for queries that filter on the partition column.")

+-----------+----------+------------------+------------------+
|pickup_hour|trip_count|          avg_fare|avg_tip_percentage|
+-----------+----------+------------------+------------------+
|         18|    206256|17.015239556667414|22.783750766291575|
|         17|    200280|18.120561114439727| 22.34330404301461|
|         16|    184947|19.459183333603697| 21.83736978978918|
|         15|    183971|19.114085535220184|19.800998170838547|
|         19|    178785|17.629133484352604|22.859468207617592|
|         14|    178002|19.273210301007893| 19.79789536000929|
|         13|    165326|18.421258967131617| 19.78590339670734|
|         12|    159884|17.798798629005994|19.742680445725988|
|         21|    155885|18.295467556211214| 21.88135250663954|
|         20|    155539|18.052625129388723|22.171556229200444|
+-----------+----------+------------------+------------------+

+-----------+----------+------------------+------------------+
|pickup_hour|trip_count|          avg_fare|avg_tip_per

Py4JJavaError: An error occurred while calling o139.parquet.
: java.lang.UnsatisfiedLinkError: 'boolean org.apache.hadoop.io.nativeio.NativeIO$Windows.access0(java.lang.String, int)'
	at org.apache.hadoop.io.nativeio.NativeIO$Windows.access0(Native Method)
	at org.apache.hadoop.io.nativeio.NativeIO$Windows.access(NativeIO.java:817)
	at org.apache.hadoop.fs.FileUtil.canRead(FileUtil.java:1415)
	at org.apache.hadoop.fs.FileUtil.list(FileUtil.java:1620)
	at org.apache.hadoop.fs.RawLocalFileSystem.listStatus(RawLocalFileSystem.java:802)
	at org.apache.hadoop.fs.FileSystem.listStatus(FileSystem.java:2078)
	at org.apache.hadoop.fs.FileSystem.listStatus(FileSystem.java:2122)
	at org.apache.hadoop.fs.ChecksumFileSystem.listStatus(ChecksumFileSystem.java:1020)
	at org.apache.hadoop.fs.FileSystem.listStatus(FileSystem.java:2078)
	at org.apache.hadoop.fs.FileSystem.listStatus(FileSystem.java:2122)
	at org.apache.hadoop.mapreduce.lib.output.FileOutputCommitter.getAllCommittedTaskPaths(F

# RAG Pipeline over Transportation Documents

In [ ]:
#import Langchain
import PyPDF2
files_memory = {}
for file in os.listdir("docs/"):
    file_paths = PyPDF2.PdfReader(os.path.join("docs/", file))
    files_memory[file] = file_paths
print ("Files read:",files_memory.keys())
for path in files_memory.values():
    for page in path.pages:
        print(f'\033[38;2;199;61;144m{file}\033[0m')
        print (page.extract_text())
        print ("----- End of page -----")

api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise ValueError("OPENAI_API_KEY environment variable is not set. Please set it to your OpenAI API key.")

Files read: dict_keys(['annual_report_2025.pdf', 'commuter_van_safety_study_2024.pdf', 'driver_expense_report.pdf', 'electrification_in_motion_report_2024.pdf', 'interior_advertising_provider_report.pdf', 'strategic_plan_2025.pdf', 'tif_report_2024.pdf'])
tif_report_2024.pdf
tif_report_2024.pdf
 
New York City Taxi and Limousine Commission  
2025 Annual Report  
January 202 6 
 
 
 

----- End of page -----
tif_report_2024.pdf
tif_report_2024.pdf
1 | 202 5 A n n u a l  R e p o r t  
 Table of Contents   
 
1. Mission and Budget  
2. Taxi and Limousine Commission Structure and Board Members  
3. Licensees Regulated by TLC  
4. Commission Meetings and Rulemaking Actions  
5. Policies, Initiatives, and Agency Highlights  
6. Appendix: Complaint and Summons Data for Calendar Year 2025  
  

----- End of page -----
tif_report_2024.pdf
tif_report_2024.pdf
2 | 202 5 A n n u a l  R e p o r t  
 Welcome  Letter from the Commissioner/Chair  
Dear Fellow New Yorkers:  
 
I am pleased to submit th

ValueError: OPENAI_API_KEY environment variable is not set. Please set it to your OpenAI API key.